In [ ]:
import sys

sys.path.append("../fracface/FracFace")
import data2npy

import os
sys.path.insert(0, "../")
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

TEST_ONLY = True
import random
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from minusface import MinusBackbone
from torch.optim.lr_scheduler import CosineAnnealingLR
import wandb
import pandas as pd
import tqdm
from insightface.app import FaceAnalysis
from datasets import load_from_disk
from tasks.partialface.utils import dct_transform, idct_transform

import sys
sys.path.insert(0, "../../../")
import cv2
from insightface.app import FaceAnalysis 
import torch


In [ ]:
device = "cuda"

In [ ]:
import torch
model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet',
    in_channels=81, out_channels=3, init_features=3, pretrained=False)

In [ ]:
tf_student = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

tf_conv = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor()
])

df = pd.read_csv('/path/to/cropped-celeba/Identity_CelebA (2).txt', sep=' ')
df.columns = ["col0","img_name","id"]

In [ ]:

embedding_root = '/path/to/casia-webface/insight_embeddings'
pre_loaded_teachers = {}
class FaceDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        # embedding_teacher = pre_loaded_teachers[p]  # (1, 512)
        img_s = cv2.imread(p)
        img_s = img_s[..., ::-1]
        img_s = Image.fromarray(img_s.astype("uint8"))
        c_img = data2npy.preprocess_and_return(img_s, 1)[0]
        
        raw_image = tf_conv(img_s)
        return p, c_img, raw_image
    

In [ ]:

paths = []
root = '/path/to/casia-webface'
with open("../fracface/index.txt","r") as f:
    lines = f.readlines()
    for line in lines:
        filename, split = line.strip().split()
        if split == "train":
            paths.append(os.path.join(root, filename))
        

dataset = FaceDataset(paths)  


val_paths = []
root = '/path/to/casia-webface'
with open("../fracface/index.txt","r") as f:
    lines = f.readlines()
    for line in lines:
        filename, split = line.strip().split()
        if split != "train":
            val_paths.append(os.path.join(root, filename)) 
        

In [ ]:
val_dataset = FaceDataset(val_paths)  

In [ ]:
print("Dataset size:", len(dataset))
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=16, pin_memory=True)
wandb.init(project="student_distill_insight") 
epochs = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=len(loader) * epochs)
os.makedirs("log", exist_ok=True)

In [ ]:
model = model.to(device)

In [ ]:
   
for e in range(epochs): 
    model = model.train()
    total_trip = 0
    total_cos = 0
    count = 0 
    total_mae = 0
    for fn, s_img, raw_image in tqdm.tqdm(loader):
        raw_image = raw_image.to(device)
        s_img = s_img.to(device)
        s_gen = model(s_img)
        loss = torch.nn.functional.l1_loss(s_gen, raw_image)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()
        count += 1
        example = torch.cat([raw_image[0].permute(1, 2, 0), s_gen[0].permute(1, 2, 0)], dim=1).cpu().detach().numpy()
        wandb.log({"train/loss": loss.item(), "train/lr": scheduler.get_last_lr()[0], "train/image": wandb.Image(example)})
    torch.save(model.state_dict(), "model_frac.pth")
    wandb.log({"epoch_triplet": total_trip/count, "epoch_cosine": total_cos/count, "epoch_mae": total_mae/count})

    model = model.eval()
    print("Validation on val set")
    total_trip = 0
    total_cos = 0
    count = 0
    total_mae = 0
    filenames = [] 
    templates = [] 
    with torch.no_grad():
        for fn, s_img, raw_image in tqdm.tqdm(val_loader):
            raw_image = raw_image.to(device)
            s_img = s_img.to(device)
            s_gen = model(s_img)
            loss = torch.nn.functional.l1_loss(s_gen, raw_image)
            optimizer.zero_grad(set_to_none=True)
            optimizer.step()
            count += 1
            example = torch.cat([raw_image[0].permute(1, 2, 0), s_gen[0].permute(1, 2, 0)], dim=1).cpu().detach().numpy()
            wandb.log({"test/loss": loss.item(), "test/image": wandb.Image(example)})


In [ ]:
torch.save(model.state_dict(), "model_frac.pth")